In [ ]:
Arrancar Spark en tu notebook.

In [1]:
import os, sys
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, count, avg, max, min, sum, desc,
    rank, broadcast, year, month, upper, split, trim, concat_ws, when, to_date
)
from pyspark.sql.window import Window

# Silenciar salida estándar de error de la JVM únicamente durante el arranque
devnull = os.open(os.devnull, os.O_WRONLY)
old_stderr = os.dup(2)
os.dup2(devnull, 2)

try:
    spark = (
        SparkSession.builder
        .appName("ED.03.03-DataFrames")
        .master("local[*]")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("ERROR")
finally:
    os.dup2(old_stderr, 2)
    os.close(devnull)
    os.close(old_stderr)

print("✅ SparkSession iniciada correctamente")
print(f"Versión de Spark: {spark.version}")

✅ SparkSession iniciada correctamente
Versión de Spark: 3.5.3


In [2]:
!mkdir -p DATOS
!wget -q "https://raw.githubusercontent.com/josepgarcia/datos/main/pdi_sales.csv" -O ./DATOS/pdi_sales.csv
!ls -lh ./DATOS/

total 38M
-rw-rw-r-- 1 valentin valentin 38M Nov 24 21:16 pdi_sales.csv


In [ ]:
Ejercicio 1 — Creación y Exploración

In [3]:
# Ejercicio 1: Creación y Exploración
datos_empleados = [
    ("Ana", 28, "Ventas", 35000),
    ("Juan", 35, "IT", 45000),
    ("María", 42, "IT", 55000),
    ("Pedro", 31, "Ventas", 38000),
    ("Laura", 29, "Marketing", 40000)
]
columnas_emp = ["nombre", "edad", "departamento", "salario"]

df_emp = spark.createDataFrame(datos_empleados, columnas_emp)

print("1. Esquema del DataFrame:")
df_emp.printSchema()

print("2. Primeras 3 filas:")
df_emp.show(3)

print(f"3. Total de empleados: {df_emp.count()}")

1. Esquema del DataFrame:
root
 |-- nombre: string (nullable = true)
 |-- edad: long (nullable = true)
 |-- departamento: string (nullable = true)
 |-- salario: long (nullable = true)

2. Primeras 3 filas:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+
only showing top 3 rows

3. Total de empleados: 5


In [ ]:
Ejercicio 2 — Filtrado con operadores lógicos (&, |, ~)

In [4]:
# Ejercicio 2: Filtrado con operadores bitwise (&, |, ~)
print("1. Empleados de IT con salario > 40000:")
df_emp.filter((col("departamento") == "IT") & (col("salario") > 40000)).show()

print("2. Empleados de Ventas O Marketing:")
df_emp.filter((col("departamento") == "Ventas") | (col("departamento") == "Marketing")).show()

print("3. Empleados que NO sean de IT:")
df_emp.filter(~(col("departamento") == "IT")).show()

print("4. Empleados entre 30 y 40 años:")
df_emp.filter((col("edad") >= 30) & (col("edad") <= 40)).show()

1. Empleados de IT con salario > 40000:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+

2. Empleados de Ventas O Marketing:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

3. Empleados que NO sean de IT:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|   Ana|  28|      Ventas|  35000|
| Pedro|  31|      Ventas|  38000|
| Laura|  29|   Marketing|  40000|
+------+----+------------+-------+

4. Empleados entre 30 y 40 años:
+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| Pedro|  31|   

In [ ]:
Ejercicio 3 — Manipulación de Columnas

In [5]:
# Ejercicio 3: Manipulación de Columnas
df_emp_proyeccion = (
    df_emp
    .withColumn("salario_en_5_años", col("salario") + (col("salario") * 0.05 * 5))
    .withColumnRenamed("nombre", "empleado")
)

df_emp_proyeccion.show()

+--------+----+------------+-------+-----------------+
|empleado|edad|departamento|salario|salario_en_5_años|
+--------+----+------------+-------+-----------------+
|     Ana|  28|      Ventas|  35000|          43750.0|
|    Juan|  35|          IT|  45000|          56250.0|
|   María|  42|          IT|  55000|          68750.0|
|   Pedro|  31|      Ventas|  38000|          47500.0|
|   Laura|  29|   Marketing|  40000|          50000.0|
+--------+----+------------+-------+-----------------+



In [ ]:
Ejercicio 4 — Agregaciones (groupBy)

In [6]:
# Ejercicio 4: Agregaciones
print("1. Salario promedio por departamento:")
df_emp.groupBy("departamento").agg(avg("salario").alias("salario_promedio")).show()

print("2. Salario máximo y mínimo por departamento:")
df_emp.groupBy("departamento").agg(
    max("salario").alias("salario_maximo"),
    min("salario").alias("salario_minimo")
).show()

print("3. Conteo de empleados por departamento:")
df_emp.groupBy("departamento").count().show()

print("4. Gasto total en salarios por departamento (ordenado de mayor a menor):")
df_emp.groupBy("departamento").agg(
    sum("salario").alias("gasto_total")
).orderBy(desc("gasto_total")).show()

1. Salario promedio por departamento:
+------------+----------------+
|departamento|salario_promedio|
+------------+----------------+
|      Ventas|         36500.0|
|          IT|         50000.0|
|   Marketing|         40000.0|
+------------+----------------+

2. Salario máximo y mínimo por departamento:
+------------+--------------+--------------+
|departamento|salario_maximo|salario_minimo|
+------------+--------------+--------------+
|      Ventas|         38000|         35000|
|          IT|         55000|         45000|
|   Marketing|         40000|         40000|
+------------+--------------+--------------+

3. Conteo de empleados por departamento:
+------------+-----+
|departamento|count|
+------------+-----+
|      Ventas|    2|
|          IT|    2|
|   Marketing|    1|
+------------+-----+

4. Gasto total en salarios por departamento (ordenado de mayor a menor):
+------------+-----------+
|departamento|gasto_total|
+------------+-----------+
|          IT|     100000|
|     

In [ ]:
Ejercicio 5 — Window Functions (Ranking por mes)

In [7]:
# Ejercicio 5: Window Functions
datos_ventas_mes = [
    ("Enero", "ProdA", 100),
    ("Enero", "ProdB", 150),
    ("Febrero", "ProdA", 120)
]
df_ventas_mes = spark.createDataFrame(datos_ventas_mes, ["mes", "producto", "ventas"])

ventana_mes = Window.partitionBy("mes").orderBy(col("ventas").desc())

df_ranking_mes = df_ventas_mes.withColumn("ranking", rank().over(ventana_mes))
df_ranking_mes.show()

+-------+--------+------+-------+
|    mes|producto|ventas|ranking|
+-------+--------+------+-------+
|  Enero|   ProdB|   150|      1|
|  Enero|   ProdA|   100|      2|
|Febrero|   ProdA|   120|      1|
+-------+--------+------+-------+



In [ ]:
Ejercicio 6 — Joins (Left Join y análisis de nulos)

In [8]:
# Ejercicio 6: Joins
df_personas = spark.createDataFrame(
    [("Ana", 1), ("Luis", 2), ("Marta", 3)],
    ["Nombre", "DepID"]
)
df_departamentos = spark.createDataFrame(
    [(1, "IT"), (2, "HR")],
    ["DepID", "NombreDep"]
)

df_left = df_personas.join(df_departamentos, "DepID", "left")
df_left.show()

print("Explicación: Como Marta tiene DepID = 3 y ese ID no existe en df_departamentos, "
      "el Left Join conserva la fila de Marta y asigna 'null' en la columna NombreDep.")

+-----+------+---------+
|DepID|Nombre|NombreDep|
+-----+------+---------+
|    1|   Ana|       IT|
|    2|  Luis|       HR|
|    3| Marta|     NULL|
+-----+------+---------+

Explicación: Como Marta tiene DepID = 3 y ese ID no existe en df_departamentos, el Left Join conserva la fila de Marta y asigna 'null' en la columna NombreDep.


In [ ]:
Ejercicio 7 — SQL Puro (createOrReplaceTempView y spark.sql)

In [9]:
# Ejercicio 7: SQL Puro
df_emp.createOrReplaceTempView("empleados")

df_sql_it = spark.sql("""
    SELECT nombre, edad, departamento, salario
    FROM empleados
    WHERE departamento = 'IT' AND salario > 40000
""")

df_sql_it.show()

+------+----+------------+-------+
|nombre|edad|departamento|salario|
+------+----+------------+-------+
|  Juan|  35|          IT|  45000|
| María|  42|          IT|  55000|
+------+----+------------+-------+



In [ ]:
Ejercicio 8 — Funciones de String (trim y concat_ws)

In [10]:
# Ejercicio 8: Funciones de String
datos_nombres = [(" Juan ", "García"), ("Ana", " Pérez ")]
df_nombres = spark.createDataFrame(datos_nombres, ["nombre", "apellido"])

df_nombres_limpio = (
    df_nombres
    .withColumn("nombre", trim(col("nombre")))
    .withColumn("apellido", trim(col("apellido")))
    .withColumn("Nombre_Completo", concat_ws(" ", col("nombre"), col("apellido")))
)

df_nombres_limpio.show()

+------+--------+---------------+
|nombre|apellido|Nombre_Completo|
+------+--------+---------------+
|  Juan|  García|    Juan García|
|   Ana|   Pérez|      Ana Pérez|
+------+--------+---------------+



In [ ]:
Cargar pdi_sales.csv, mostrar esquema y contar filas

In [11]:
# Proyecto: Análisis de Ventas - Paso 1: Carga, esquema y conteo
df_sales = spark.read.csv(
    "DATOS/pdi_sales.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

print("1. Esquema del dataset pdi_sales.csv:")
df_sales.printSchema()

print(f"2. Total de registros (filas): {df_sales.count()}")
df_sales.show(5)

1. Esquema del dataset pdi_sales.csv:
root
 |-- ProductID: integer (nullable = true)
 |-- Date: string (nullable = true)
 |-- Zip: string (nullable = true)
 |-- Units: integer (nullable = true)
 |-- Revenue: double (nullable = true)
 |-- Country: string (nullable = true)

2. Total de registros (filas): 841147
+---------+---------+---------------+-----+-------+-------+
|ProductID|     Date|            Zip|Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|      725|1/15/1999|41540          |    1|  115.5|Germany|
|      787| 6/6/2002|41540          |    1|  314.9|Germany|
|      788| 6/6/2002|41540          |    1|  314.9|Germany|
|      940|1/15/1999|22587          |    1|  687.7|Germany|
|      396|1/15/1999|22587          |    1|  857.1|Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 5 rows



In [ ]:
Total de ingresos (Revenue) por país (Country) ordenado de mayor a menor

In [12]:
# Proyecto: Análisis de Ventas - Paso 2: Ingresos totales por país
from pyspark.sql.functions import round as spark_round

ingresos_pais = (
    df_sales
    .groupBy("Country")
    .agg(spark_round(sum("Revenue"), 2).alias("Total_Revenue"))
    .orderBy(desc("Total_Revenue"))
)

ingresos_pais.show()

+-------+-------------+
|Country|Total_Revenue|
+-------+-------------+
|France |1.484901933E8|
|Germany|1.319305101E8|
|Mexico |1.036458026E8|
|Canada | 3.41149309E7|
+-------+-------------+



In [ ]:
Filtrar ventas de Alemania (Germany) con más de 1 unidad (Units > 1)

In [13]:
# Proyecto: Análisis de Ventas - Paso 3: Ventas en Alemania con Units > 1
ventas_alemania_gt1 = df_sales.filter((col("Country") == "Germany") & (col("Units") > 1))

print(f"Total de ventas en Alemania con más de 1 unidad: {ventas_alemania_gt1.count()}")
ventas_alemania_gt1.show(10)

Total de ventas en Alemania con más de 1 unidad: 7426
+---------+---------+---------------+-----+-------+-------+
|ProductID|     Date|            Zip|Units|Revenue|Country|
+---------+---------+---------------+-----+-------+-------+
|      901|2/15/1999|13587          |    2|  818.9|Germany|
|      676|1/15/1999|21107          |    2| 1603.9|Germany|
|      604|4/10/2003|10243          |    2|  989.6|Germany|
|      491|3/15/1999|10319          |    2| 1630.7|Germany|
|      910|3/15/1999|10435          |    2|  829.4|Germany|
|      645|3/15/1999|10435          |    2| 1493.1|Germany|
|      862|4/10/2003|10435          |    2|  608.9|Germany|
|      499|2/15/1999|10435          |    2| 1692.6|Germany|
|      631|2/15/1999|40880          |    2|  692.5|Germany|
|      635| 6/3/2002|12627          |    2| 1716.2|Germany|
+---------+---------+---------------+-----+-------+-------+
only showing top 10 rows



In [ ]:
Código postal (Zip) con mayor número de ventas en Alemania

In [14]:
# Proyecto: Análisis de Ventas - Paso 4: Código postal (Zip) con más ventas en Alemania
top_zip_alemania = (
    df_sales
    .filter(col("Country") == "Germany")
    .groupBy("Zip")
    .agg(count("*").alias("Numero_Ventas"))
    .orderBy(desc("Numero_Ventas"))
)

print("Top 5 códigos postales con más ventas en Alemania:")
top_zip_alemania.show(5)

Top 5 códigos postales con más ventas en Alemania:
+---------------+-------------+
|            Zip|Numero_Ventas|
+---------------+-------------+
|50169          |         2130|
|52538          |         2007|
|40597          |         1980|
|20251          |         1828|
|22111          |         1780|
+---------------+-------------+
only showing top 5 rows



In [ ]:
Precio unitario promedio (Precio_Unitario) y Top 5 productos con mayores ingresos totales (ProductID)

In [15]:
# Proyecto: Análisis de Ventas - Paso 5: Precio unitario y Top 5 productos por ingresos
df_sales_unitario = df_sales.withColumn(
    "Precio_Unitario",
    spark_round(col("Revenue") / col("Units"), 2)
)

print("Muestra con columna Precio_Unitario:")
df_sales_unitario.select("ProductID", "Country", "Units", "Revenue", "Precio_Unitario").show(5)

top_productos = (
    df_sales_unitario
    .groupBy("ProductID")
    .agg(
        sum("Units").alias("Total_Unidades"),
        spark_round(sum("Revenue"), 2).alias("Ingresos_Totales")
    )
    .orderBy(desc("Ingresos_Totales"))
)

print("Top 5 productos con mayores ingresos totales:")
top_productos.show(5)

Muestra con columna Precio_Unitario:
+---------+-------+-----+-------+---------------+
|ProductID|Country|Units|Revenue|Precio_Unitario|
+---------+-------+-----+-------+---------------+
|      725|Germany|    1|  115.5|          115.5|
|      787|Germany|    1|  314.9|          314.9|
|      788|Germany|    1|  314.9|          314.9|
|      940|Germany|    1|  687.7|          687.7|
|      396|Germany|    1|  857.1|          857.1|
+---------+-------+-----+-------+---------------+
only showing top 5 rows

Top 5 productos con mayores ingresos totales:
+---------+--------------+----------------+
|ProductID|Total_Unidades|Ingresos_Totales|
+---------+--------------+----------------+
|      475|         12335|    1.26670973E7|
|      487|         11329|    1.23163334E7|
|      826|          9043|       9378033.5|
|      506|          7172|       9244457.2|
|      438|          9377|       9172885.1|
+---------+--------------+----------------+
only showing top 5 rows

